# ASHBOURNE & CO.
##### _________________________________________________________________________________________________________________________________________________________________________________________

####
### 1. IMPORT LIBRARIES

In [1]:
import pyodbc
import pandas as pd

####
### 2. SQL SERVER CONNECTION DETAILS

In [2]:
SERVER = r"DESKTOP-0SIEUH6\SQLEXPRESS"
DATABASE = "AshbourneCo"

####
### 3. CREATE DATABASE

In [3]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER};"
    "DATABASE=master;"
    "Trusted_Connection=yes;",
    autocommit=True
)

cursor = conn.cursor()

cursor.execute("""
IF DB_ID('AshbourneCo') IS NULL
BEGIN
    CREATE DATABASE AshbourneCo;
END
""")

print("Database 'AshbourneCo' is ready.")

cursor.close()
conn.close()

Database 'AshbourneCo' is ready.


####
### 4. CONNECT TO AshbourneCo

In [4]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()

print("Connected to AshbourneCo successfully.")


Connected to AshbourneCo successfully.


####
### 5. LOAD CSV FILES

In [5]:
customers = pd.read_csv(
    r"C:\Users\Naveen\Desktop\GitHub\Ashbourne & Co\dataset\Ashbourne_Customers.csv"
)

inventory = pd.read_csv(
    r"C:\Users\Naveen\Desktop\GitHub\Ashbourne & Co\dataset\Ashbourne_Inventory.csv"
)

marketing = pd.read_csv(
    r"C:\Users\Naveen\Desktop\GitHub\Ashbourne & Co\dataset\Ashbourne_Marketing_Campaigns.csv"
)

orders = pd.read_csv(
    r"C:\Users\Naveen\Desktop\GitHub\Ashbourne & Co\dataset\Ashbourne_Orders.csv"
)

products = pd.read_csv(
    r"C:\Users\Naveen\Desktop\GitHub\Ashbourne & Co\dataset\Ashbourne_Products.csv"
)

stores = pd.read_csv(
    r"C:\Users\Naveen\Desktop\GitHub\Ashbourne & Co\dataset\Ashbourne_Stores.csv"
)


print("\nCSV files loaded successfully.")

print(f"Customers            : {len(customers):,}")
print(f"Products             : {len(products):,}")
print(f"Stores               : {len(stores):,}")
print(f"Marketing Campaigns  : {len(marketing):,}")
print(f"Orders               : {len(orders):,}")
print(f"Inventory            : {len(inventory):,}")



CSV files loaded successfully.
Customers            : 3,000
Products             : 64
Stores               : 16
Marketing Campaigns  : 34
Orders               : 26,500
Inventory            : 1,024


####
### 6. DROP EXISTING TABLES

In [6]:
cursor.execute("""
IF OBJECT_ID('dbo.inventory', 'U') IS NOT NULL
    DROP TABLE dbo.inventory;

IF OBJECT_ID('dbo.orders', 'U') IS NOT NULL
    DROP TABLE dbo.orders;

IF OBJECT_ID('dbo.marketing_campaigns', 'U') IS NOT NULL
    DROP TABLE dbo.marketing_campaigns;

IF OBJECT_ID('dbo.stores', 'U') IS NOT NULL
    DROP TABLE dbo.stores;

IF OBJECT_ID('dbo.products', 'U') IS NOT NULL
    DROP TABLE dbo.products;

IF OBJECT_ID('dbo.customers', 'U') IS NOT NULL
    DROP TABLE dbo.customers;
""")

conn.commit()

print("\nExisting AshbourneCo tables removed.")



Existing AshbourneCo tables removed.


####
### 7. CREATE TABLES

In [7]:

# ------------------------------------------------------------
# 7.1 CUSTOMERS
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    gender VARCHAR(20),
    age INT,
    income_bracket VARCHAR(30),
    acquisition_source VARCHAR(50),
    city VARCHAR(50),
    join_date DATE,
    loyalty_tier VARCHAR(20)
);
""")


# ------------------------------------------------------------
# 7.2 PRODUCTS
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(150),
    category VARCHAR(20),
    sub_category VARCHAR(50),
    season VARCHAR(30),
    material VARCHAR(50),
    unit_cost DECIMAL(12,2),
    retail_price DECIMAL(12,2)
);
""")


# ------------------------------------------------------------
# 7.3 STORES
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE stores (
    store_id INT PRIMARY KEY,
    store_name VARCHAR(100),
    city VARCHAR(50),
    store_type VARCHAR(30),
    square_footage INT,
    open_date DATE
);
""")


# ------------------------------------------------------------
# 7.4 MARKETING CAMPAIGNS
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE marketing_campaigns (
    campaign_id INT PRIMARY KEY,
    campaign_name VARCHAR(150),
    channel VARCHAR(50),
    budget DECIMAL(12,2),
    start_date DATE,
    end_date DATE
);
""")


# ------------------------------------------------------------
# 7.5 INVENTORY
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE inventory (
    inventory_id INT PRIMARY KEY,
    product_id INT NOT NULL,
    store_id INT NOT NULL,
    stock_level INT,
    last_restock_date DATE,

    CONSTRAINT FK_inventory_products
        FOREIGN KEY (product_id)
        REFERENCES products(product_id),

    CONSTRAINT FK_inventory_stores
        FOREIGN KEY (store_id)
        REFERENCES stores(store_id)
);
""")


# ------------------------------------------------------------
# 7.6 ORDERS — CENTRAL FACT TABLE
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    product_id INT NOT NULL,
    store_id INT NOT NULL,
    campaign_id INT NULL,

    order_date DATE,
    quantity INT,
    discount_pct DECIMAL(5,4),
    gross_sales DECIMAL(12,2),
    discount_amount DECIMAL(12,2),
    net_sales DECIMAL(12,2),
    cogs DECIMAL(12,2),
    profit DECIMAL(12,2),

    return_status VARCHAR(20),
    rating INT,
    delivery_date DATE,

    CONSTRAINT FK_orders_customers
        FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id),

    CONSTRAINT FK_orders_products
        FOREIGN KEY (product_id)
        REFERENCES products(product_id),

    CONSTRAINT FK_orders_stores
        FOREIGN KEY (store_id)
        REFERENCES stores(store_id),

    CONSTRAINT FK_orders_campaigns
        FOREIGN KEY (campaign_id)
        REFERENCES marketing_campaigns(campaign_id)
);
""")


conn.commit()

print("\nAll Ashbourne & Co. tables created successfully.")


All Ashbourne & Co. tables created successfully.


####
### 8. INSERT ALL CSV DATA INTO SQL SERVER

In [8]:
# ------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------

def clean_value(value):

    # Convert Pandas missing values to SQL NULL
    if pd.isna(value):
        return None

    # Convert NumPy/Pandas numeric types to native Python types
    if hasattr(value, "item"):
        try:
            return value.item()
        except:
            pass

    return value


def insert_dataframe(
    cursor,
    df,
    table_name,
    columns,
    date_columns=None,
    batch_size=5000
):

    data = df.copy()

    # --------------------------------------------------------
    # CONVERT DATE COLUMNS
    # --------------------------------------------------------

    if date_columns:

        for col in date_columns:

            data[col] = pd.to_datetime(
                data[col],
                errors="coerce"
            ).dt.date


    # --------------------------------------------------------
    # CONVERT NaN / NaT TO NONE
    # --------------------------------------------------------

    data = data.astype(object)

    data = data.where(
        pd.notnull(data),
        None
    )


    # --------------------------------------------------------
    # CREATE SQL INSERT STATEMENT
    # --------------------------------------------------------

    column_names = ", ".join(columns)

    placeholders = ", ".join(
        ["?"] * len(columns)
    )

    sql = f"""
        INSERT INTO {table_name}
        ({column_names})
        VALUES ({placeholders})
    """


    # --------------------------------------------------------
    # CONVERT DATAFRAME TO PYTHON VALUES
    # --------------------------------------------------------

    values = []

    for row in data[columns].itertuples(
        index=False,
        name=None
    ):

        cleaned_row = tuple(
            clean_value(value)
            for value in row
        )

        values.append(cleaned_row)


    # --------------------------------------------------------
    # INSERT IN BATCHES
    # --------------------------------------------------------

    cursor.fast_executemany = True

    total_rows = len(values)

    for start in range(
        0,
        total_rows,
        batch_size
    ):

        batch = values[
            start:start + batch_size
        ]

        cursor.executemany(
            sql,
            batch
        )

        conn.commit()

        inserted = min(
            start + batch_size,
            total_rows
        )

        print(
            f"{table_name:<25} : "
            f"{inserted:,} / {total_rows:,}"
        )


# ============================================================
# 8.1 CUSTOMERS
# ============================================================

insert_dataframe(
    cursor,
    customers,
    "customers",

    [
        "customer_id",
        "gender",
        "age",
        "income_bracket",
        "acquisition_source",
        "city",
        "join_date",
        "loyalty_tier"
    ],

    date_columns=[
        "join_date"
    ]
)


# ============================================================
# 8.2 PRODUCTS
# ============================================================

insert_dataframe(
    cursor,
    products,
    "products",

    [
        "product_id",
        "product_name",
        "category",
        "sub_category",
        "season",
        "material",
        "unit_cost",
        "retail_price"
    ]
)


# ============================================================
# 8.3 STORES
# ============================================================

insert_dataframe(
    cursor,
    stores,
    "stores",

    [
        "store_id",
        "store_name",
        "city",
        "store_type",
        "square_footage",
        "open_date"
    ],

    date_columns=[
        "open_date"
    ]
)


# ============================================================
# 8.4 MARKETING CAMPAIGNS
# ============================================================

insert_dataframe(
    cursor,
    marketing,
    "marketing_campaigns",

    [
        "campaign_id",
        "campaign_name",
        "channel",
        "budget",
        "start_date",
        "end_date"
    ],

    date_columns=[
        "start_date",
        "end_date"
    ]
)


# ============================================================
# 8.5 INVENTORY
# ============================================================

insert_dataframe(
    cursor,
    inventory,
    "inventory",

    [
        "inventory_id",
        "product_id",
        "store_id",
        "stock_level",
        "last_restock_date"
    ],

    date_columns=[
        "last_restock_date"
    ]
)


# ============================================================
# 8.6 ORDERS
# ============================================================

insert_dataframe(
    cursor,
    orders,
    "orders",

    [
        "order_id",
        "customer_id",
        "product_id",
        "store_id",
        "campaign_id",
        "order_date",
        "quantity",
        "discount_pct",
        "gross_sales",
        "discount_amount",
        "net_sales",
        "cogs",
        "profit",
        "return_status",
        "rating",
        "delivery_date"
    ],

    date_columns=[
        "order_date",
        "delivery_date"
    ],

    batch_size=5000
)


# ============================================================
# FINAL COMMIT
# ============================================================

conn.commit()


# ============================================================
# INSERTION COMPLETE
# ============================================================

print("\n")
print("=" * 60)
print("ALL DATA INSERTED SUCCESSFULLY")
print("=" * 60)

print(f"Customers            : {len(customers):,}")
print(f"Products             : {len(products):,}")
print(f"Stores               : {len(stores):,}")
print(f"Marketing Campaigns  : {len(marketing):,}")
print(f"Inventory            : {len(inventory):,}")
print(f"Orders               : {len(orders):,}")

print("=" * 60)

customers                 : 3,000 / 3,000
products                  : 64 / 64
stores                    : 16 / 16
marketing_campaigns       : 34 / 34
inventory                 : 1,024 / 1,024
orders                    : 5,000 / 26,500
orders                    : 10,000 / 26,500
orders                    : 15,000 / 26,500
orders                    : 20,000 / 26,500
orders                    : 25,000 / 26,500
orders                    : 26,500 / 26,500


ALL DATA INSERTED SUCCESSFULLY
Customers            : 3,000
Products             : 64
Stores               : 16
Marketing Campaigns  : 34
Inventory            : 1,024
Orders               : 26,500


# ---------------------------- Database Created sucessfully ----------------------------